In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from collections import defaultdict
from itertools import product


In [ ]:
# Path to your .ply file
file_path = '/storage/user/ayu/repos/HI-SLAM2/outputs/semantic/room0_17/3dgs_final.ply'

# Define the expected fields and corresponding NumPy dtypes
fields = [
    'x', 'y', 'z',
    'nx', 'ny', 'nz',
    'f_dc_0', 'f_dc_1', 'f_dc_2',
    'opacity',
    'ins_feat_0', 'ins_feat_1', 'ins_feat_2',
    'ins_feat_3', 'ins_feat_4', 'ins_feat_5',
    'scale_0', 'scale_1', 'scale_2',
    'rot_0', 'rot_1', 'rot_2', 'rot_3'
]

dtype = np.dtype([(name, 'f4') for name in fields])

# Read header and find vertex count and byte offset
with open(file_path, 'rb') as f:
    header_lines = []
    while True:
        line = f.readline()
        header_lines.append(line)
        if line.strip() == b'end_header':
            break

    # Decode and search for vertex count
    vertex_count = 0
    for line in header_lines:
        if line.startswith(b'element vertex'):
            vertex_count = int(line.decode().split()[-1])
            break

    # Sanity check
    if vertex_count == 0:
        raise ValueError("Could not determine number of vertices from header.")

    # Read binary data
    data = np.fromfile(f, dtype=dtype, count=vertex_count)

# Structured array is now loaded
array_2d = np.vstack([data[field] for field in fields]).T
print(f"Loaded {len(data)} vertices.")
print(array_2d[0])
pos = torch.Tensor(array_2d[:, :3])  # Extract position data (x, y, z)
normals = torch.Tensor(array_2d[:, 3:6])  # Extract normal data
spherical_harmonics = torch.Tensor(array_2d[:, 6:9])  # Extract spherical harmonics data
opacity = torch.Tensor(array_2d[:, 9])  # Extract opacity data
ins_feats = torch.Tensor(array_2d[:, 10:16])
ins_feats = (ins_feats - ins_feats.min(dim=0, keepdim=True).values) / (ins_feats.max(dim=0, keepdim=True).values - ins_feats.min(dim=0, keepdim=True).values + 1e-8)
scale = torch.Tensor(array_2d[:, 16:19])  # Extract scale data
rot = torch.Tensor(array_2d[:, 19:23])  # Extract rotation data

In [ ]:
from hislam2.cluster_gaussians import over_segmentation
num_clusters = 1024
labels = over_segmentation(pos, ins_feats, num_clusters)
X = pos.numpy()
labels = labels.numpy()

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=labels, cmap='viridis', s=50)
ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('Z axis')

In [ ]:
# df = pd.DataFrame(X, columns=['x', 'y', 'z'])
# df['cluster'] = labels.astype(str)
# fig = px.scatter_3d(df, x='x', y='y', z='z', color='cluster',
#                     title="3D Clustering Visualization (Plotly)",
#                     width=1024, height=1024)  # <- Adjust size here
# fig.update_traces(marker=dict(size=4))  # Adjust size as needed
# fig.show()

In [ ]:
def group_clusters(
    features: torch.Tensor,
    labels: torch.Tensor
):
    """
    Groups features into clusters based on labels.

    Args:
        features (torch.Tensor): Tensor of shape (N, D)
        labels (torch.Tensor): Tensor of shape (N,) with cluster indices

    Returns:
        List[Dict[str, torch.Tensor]]: A list where each element is a dict with keys:
            'avg_feature' -> (D,)
            'indexes' -> (K_i,)
    """
    clusters = []
    labels_tensor = torch.tensor(labels, dtype=torch.int64)
    num_clusters = int(labels_tensor.max().item() + 1)

    for i in range(num_clusters):
        mask = (labels_tensor == i)
        cluster_features = features[mask]
        clusters.append({
            'avg_feature': cluster_features.mean(dim=0),
            'indexes': torch.where(mask)[0]
        })

    return clusters


In [ ]:
def build_adjacency_matrix(points: np.ndarray, labels: np.ndarray, features: torch.Tensor, resolution: int = 128):
    """
    Builds adjacency matrix based on voxel proximity and feature similarity.
    """

    # Normalize points to voxel grid
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler(feature_range=(0, resolution - 1))
    points_scaled = scaler.fit_transform(points).astype(np.int32)

    # Build voxel grid
    voxel_grid = defaultdict(set)
    for p, lbl in zip(points_scaled, labels):
        voxel_grid[tuple(p)].add(int(lbl))

    # Neighbor voxel offsets
    neighbor_offsets = [offset for offset in product([-1, 0, 1], repeat=3) if offset != (0, 0, 0)]

    # Initialize adjacency matrix
    K = int(labels.max()) + 1
    adj_matrix = torch.zeros((K, K), dtype=torch.uint8)

    # Group clusters and compute average features
    gaussians_tensor = torch.tensor(points, dtype=torch.float32)
    labels_tensor = torch.tensor(labels, dtype=torch.int64)
    clusters = group_clusters(features, labels_tensor)

    # Precompute avg_features
    avg_features = [c['avg_feature'] for c in clusters]

    # For quick L2 check
    def is_similar(c1, c2):
        f1 = avg_features[c1]
        f2 = avg_features[c2]
        return torch.norm(f1 - f2).item() < 0.2

    for voxel, cluster_set in voxel_grid.items():
        cluster_list = list(cluster_set)

        # Same voxel
        for i in range(len(cluster_list)):
            for j in range(i + 1, len(cluster_list)):
                c1, c2 = int(cluster_list[i]), int(cluster_list[j])
                if is_similar(c1, c2):
                    adj_matrix[c1, c2] = 1
                    adj_matrix[c2, c1] = 1

        # Neighboring voxels
        x, y, z = voxel
        for dx, dy, dz in neighbor_offsets:
            neighbor_voxel = (x + dx, y + dy, z + dz)
            if neighbor_voxel in voxel_grid:
                for c1 in cluster_set:
                    for c2 in voxel_grid[neighbor_voxel]:
                        c1, c2 = int(c1), int(c2)
                        if c1 != c2 and is_similar(c1, c2):
                            adj_matrix[c1, c2] = 1
                            adj_matrix[c2, c1] = 1

    return adj_matrix

In [ ]:


# points = pos.numpy()
# scaler = MinMaxScaler(feature_range=(0, 127))
# points_scaled = scaler.fit_transform(points).astype(np.int32)

In [ ]:
# voxel_grid = defaultdict(set)

# for point, label in zip(points_scaled, labels):
#     voxel = tuple(point)  # (x, y, z)
#     voxel_grid[voxel].add(label)

In [ ]:

# adj_matrix = torch.zeros((num_clusters, num_clusters), dtype=torch.uint8)

# # 26 neighboring directions (3x3x3 cube minus center)
# neighbor_offsets = [offset for offset in product([-1, 0, 1], repeat=3) if offset != (0, 0, 0)]

# for voxel, cluster_set in voxel_grid.items():
#     cluster_list = list(cluster_set)

#     # Check all pairs in the same voxel
#     for i in range(len(cluster_list)):
#         for j in range(i + 1, len(cluster_list)):
#             c1, c2 = cluster_list[i], cluster_list[j]
#             c1 = int(c1)
#             c2 = int(c2)
#             adj_matrix[c1, c2] = 1
#             adj_matrix[c2, c1] = 1

#     # Check neighboring voxels
#     x, y, z = voxel
#     for dx, dy, dz in neighbor_offsets:
#         neighbor_voxel = (x + dx, y + dy, z + dz)
#         if neighbor_voxel in voxel_grid:
#             for c1 in cluster_set:
#                 for c2 in voxel_grid[neighbor_voxel]:
#                     if c1 != c2:
#                         c1 = int(c1)
#                         c2 = int(c2)
#                         adj_matrix[c1, c2] = 1
#                         adj_matrix[c2, c1] = 1


In [ ]:
clusters = group_clusters(ins_feats, labels)

In [ ]:
# def bounding_boxes_intersect(min_a, max_a, min_b, max_b) -> bool:
#     return torch.all(max_a >= min_b) and torch.all(max_b >= min_a)

In [ ]:
# def compute_adjacency_matrix(clusters: list) -> torch.Tensor:
#     """
#     Computes an adjacency matrix for clusters based on bounding box intersections.

#     Args:
#         clusters (List[Dict]): Each dict must contain 'gaussians': (K, 3)

#     Returns:
#         torch.Tensor: Adjacency matrix of shape (num_clusters, num_clusters) with 1s where clusters intersect.
#     """
#     num_clusters = len(clusters)
#     adj_matrix = torch.zeros((num_clusters, num_clusters), dtype=torch.uint8)

#     # Precompute bounding boxes
#     bboxes = []
#     for c in clusters:
#         g = c['gaussians']
#         min_corner = g.min(dim=0).values
#         max_corner = g.max(dim=0).values
#         bboxes.append((min_corner, max_corner))

#     # Compare all pairs
#     for i in range(num_clusters):
#         for j in range(i + 1, num_clusters):
#             min_i, max_i = bboxes[i]
#             min_j, max_j = bboxes[j]
#             if bounding_boxes_intersect(min_i, max_i, min_j, max_j) and (clusters[i]['avg_feature'] - clusters[j]['avg_feature']).norm() < 0.1:
#                 adj_matrix[i, j] = 1
#                 adj_matrix[j, i] = 1  # Symmetric

#     return adj_matrix

In [ ]:
# adj_matrix = compute_adjacency_matrix(clusters)

In [ ]:
def connected_components(adj_matrix: torch.Tensor):
    """
    Computes connected components from a binary adjacency matrix.

    Args:
        adj_matrix (torch.Tensor): Shape (N, N), with 1s indicating edges.

    Returns:
        List[List[int]]: A list of components; each component is a list of node indices.
    """
    num_nodes = adj_matrix.shape[0]
    visited = torch.zeros(num_nodes, dtype=torch.bool)
    components = []

    for i in range(num_nodes):
        if not visited[i]:
            stack = [i]
            component = []

            while stack:
                node = stack.pop()
                if not visited[node]:
                    visited[node] = True
                    component.append(node)
                    neighbors = (adj_matrix[node] == 1).nonzero(as_tuple=False).flatten().tolist()
                    stack.extend(neighbors)

            components.append(component)

    return components


In [ ]:
def merge_clusters_by_components(clusters, components):
    merged_clusters = []

    for comp in components:
        indexes = []

        for i in comp:
            i = int(i)  # Ensure Python int
            idx = clusters[i]['indexes']

            # Convert list, tuple, or tensor to torch.LongTensor
            if isinstance(idx, (list, tuple)):
                idx = torch.tensor(idx, dtype=torch.long)
            elif isinstance(idx, torch.Tensor):
                idx = idx.to(dtype=torch.long)
            else:
                raise TypeError(f"Unsupported index type: {type(idx)}")

            indexes.append(idx)

        merged_indexes = torch.cat(indexes, dim=0)

        merged_clusters.append({
            'indexes': merged_indexes,
        })

    return merged_clusters


In [ ]:
adj_matrix = build_adjacency_matrix(pos.numpy(), labels, ins_feats)
components = connected_components(adj_matrix)


In [ ]:
merged_clusters = merge_clusters_by_components(clusters, components)
labels = torch.tensor(labels, dtype=torch.long)  # ← Fix here
new_labels = labels.clone()
for i, c in enumerate(merged_clusters):
    new_labels[c['indexes']] = i


In [ ]:
X = pos.numpy()
new_labels = new_labels.numpy()
df = pd.DataFrame(X, columns=['x', 'y', 'z'])
df['cluster'] = new_labels.astype(str)
fig = px.scatter_3d(df, x='x', y='y', z='z', color='cluster',
                    title="3D Clustering Visualization (Plotly)",
                    width=1024, height=1024)  # <- Adjust size here
fig.update_traces(marker=dict(size=4))  # Adjust size as needed
fig.show()